In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
%run ../00-common-Config/02-helper-function

In [0]:
landing_path
source_file=f"{landing_path}/races.csv"
table_name=f"{catalog_name}.{bronze_schema}.races"

##creating the bronze table for race table


In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType, DateType

race_schema=StructType(
    [StructField('season', IntegerType(), True), 
    StructField('round', IntegerType(), True),
    StructField('url', StringType(), True), 
    StructField('raceName', StringType(), True), 
    StructField('date', DateType(), True), 
    StructField('circuitId', StringType(), True)])

In [0]:
race_df=spark.read.format("csv")\
    .option("header","true")\
    .schema(race_schema)\
    .option("mode","FAILFAST")\
    .load(source_file)
race_df.show()

##adding source file name column and ingestion_timestamp


In [0]:
# from pyspark.sql import functions as F
# # race_df=race_df.withColumn("ingestion_timestamp", F.current_timestamp())
# # race_df=race_df.withColumn("source_file", F.col('_metadata.file_path'))
race_df=add_ingestion_metadata(race_df)
race_df.show()


###creating bronze table for race table

In [0]:
race_df.write.format("delta")\
    .mode("overwrite")\
    .saveAsTable(table_name)

checking the record in race table

In [0]:
spark.read.table(table_name).show()